# Phase 9 ? Post-hoc Model C prediction on observed ViLexNorm Test

> **Warning:** This is a descriptive post-hoc benchmark. It uses the previously observed Phase 5 Test and cannot promote Model C or modify Model B's selection.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

REPO = Path('/kaggle/working/VisolexNorm')
SOURCE_REF = 'main'
REPOSITORY_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/VisolexNorm.git'

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', SOURCE_REF, REPOSITORY_URL, str(REPO)], check=True)
source_commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', source_commit], check=True)
print(f'Running Phase 9 source at {source_commit}')


In [ ]:
%cd {REPO}
!pip install -q -r requirements-kaggle.txt
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
print(torch.cuda.get_device_name(0))


## Frozen inputs
Attach datasets containing the Phase 5 raw A/B predictions and provenance, the Model C checkpoint and Phase 8 outputs, the processed ViLexNorm Test, and the locally frozen Phase 9 benchmark manifest.


In [ ]:
DATA = Path('/kaggle/input/visolexnorm-processed')
MODEL_A = Path('/kaggle/input/phase2-output/checkpoints/model_a')
MODEL_B = Path('/kaggle/input/phase4-output/checkpoints/model_b')
MODEL_C = Path('/kaggle/input/phase8-output/checkpoints/model_c')
PHASE4 = Path('/kaggle/input/phase4-output/outputs/model_b/phase4_exit_report.json')
PHASE5 = Path('/kaggle/input/phase5-evaluation')
PHASE8 = Path('/kaggle/input/phase8-output/outputs/model_c')
BENCHMARK = Path('/kaggle/input/phase9-posthoc/benchmark_manifest.json')
WORK = Path('/kaggle/working/evaluation_abc_posthoc')

TEST = DATA / 'data/processed/vilexnorm_test.jsonl'
PHASE3_MANIFEST = DATA / 'outputs/phase3_manifest.json'
CONFIG = REPO / 'configs/evaluation_generation_config.json'
METRIC_CODE = REPO / 'scripts/evaluation_metrics.py'
SCHEMA = REPO / 'specs/009-posthoc-abc-benchmark/contracts/prediction.schema.json'
A_PREDICTION = PHASE5 / 'model_a_test_predictions.jsonl'
B_PREDICTION = PHASE5 / 'model_b_test_predictions.jsonl'
PHASE5_MANIFEST = PHASE5 / 'freeze_manifest.json'
PHASE5_METRICS = PHASE5 / 'test_metrics.json'
C_ARTIFACT = PHASE8 / 'artifact_manifest.json'
C_EXIT = PHASE8 / 'phase8_exit_report.json'
required = (MODEL_A, MODEL_B, MODEL_C, PHASE4, TEST, PHASE3_MANIFEST, CONFIG, METRIC_CODE, SCHEMA, A_PREDICTION, B_PREDICTION, PHASE5_MANIFEST, PHASE5_METRICS, C_ARTIFACT, C_EXIT, BENCHMARK)
missing = [str(path) for path in required if not path.exists()]
assert not missing, 'Missing frozen input(s):\n' + '\n'.join(missing)
WORK.mkdir(parents=True, exist_ok=True)


In [ ]:
def run_checked(*command: str) -> None:
    print('$', ' '.join(command))
    subprocess.run(command, cwd=REPO, check=True)

run_checked('python', '-m', 'scripts.evaluation', 'posthoc-verify', '--manifest', str(BENCHMARK), '--phase5-manifest', str(PHASE5_MANIFEST), '--phase5-metrics', str(PHASE5_METRICS), '--model-a-prediction', str(A_PREDICTION), '--model-b-prediction', str(B_PREDICTION), '--model-a-checkpoint', str(MODEL_A), '--model-b-checkpoint', str(MODEL_B), '--model-c-checkpoint', str(MODEL_C), '--model-c-artifact-manifest', str(C_ARTIFACT), '--model-c-exit-report', str(C_EXIT), '--test', str(TEST), '--generation-config', str(CONFIG), '--metric-code', str(METRIC_CODE), '--phase3-manifest', str(PHASE3_MANIFEST), '--phase4-exit-report', str(PHASE4), '--schema', str(SCHEMA))


In [ ]:
C_PREDICTION = WORK / 'model_c_test_predictions.jsonl'
run_checked('python', '-m', 'scripts.evaluation', 'posthoc-generate', '--manifest', str(BENCHMARK), '--model', 'model_c', '--checkpoint', str(MODEL_C), '--model-c-checkpoint', str(MODEL_C), '--output', str(C_PREDICTION), '--phase5-manifest', str(PHASE5_MANIFEST), '--phase5-metrics', str(PHASE5_METRICS), '--model-a-prediction', str(A_PREDICTION), '--model-b-prediction', str(B_PREDICTION), '--model-a-checkpoint', str(MODEL_A), '--model-b-checkpoint', str(MODEL_B), '--model-c-artifact-manifest', str(C_ARTIFACT), '--model-c-exit-report', str(C_EXIT), '--test', str(TEST), '--generation-config', str(CONFIG), '--metric-code', str(METRIC_CODE), '--phase3-manifest', str(PHASE3_MANIFEST), '--phase4-exit-report', str(PHASE4), '--schema', str(SCHEMA))
rows = [json.loads(line) for line in C_PREDICTION.read_text(encoding='utf-8').splitlines()]
assert len(rows) == 1045 and len({row['id'] for row in rows}) == 1045
assert all(row['model'] == 'model_c' for row in rows)


In [ ]:
shutil.make_archive('/kaggle/working/model_c_posthoc_prediction', 'zip', WORK)
print('Download model_c_posthoc_prediction.zip. Score it locally; do not retrain or modify Model C after this result.')
